# **LLM Training Data Preparation**

In [2]:
from datasets import load_dataset

wiki_dataset = load_dataset("wikitext", "wikitext-103-v1")
wiki_dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [8]:
training_data = [data["text"] for data in wiki_dataset["train"] if len(data["text"]) > 10]
len(training_data)

1160832

In [29]:
import json

def collect_vocab(list_training_data: list):
    combined_string = " ".join(list_training_data)

    list_vocab = list(set(combined_string.split()))

    vocab_json = {vocab: idx for idx, vocab in enumerate(list_vocab, start=1)}
    vocab_json["<EOS>"] = 0

    with open("vocab.json", "w", encoding="utf-8") as f:
        json.dump(vocab_json, f, indent=4, ensure_ascii=False)

    return vocab_json

In [30]:
vocab_json = collect_vocab(training_data)

In [63]:
len(vocab_json)

267723

## **Prepare for the training data**

In [58]:
import torch

with open("vocab.json", "r") as openfile:
    vocab_json = json.loads(openfile.read())

def prepare_training_data(list_training_data: list):
    list_input_data = []
    list_label_data = []
    combined_text = " <EOS> ".join(list_training_data)
    token_ids_input = [vocab_json[token] for token in combined_text.split()]
    total_data = len(token_ids_input) // 256
    for idx in range(total_data):
        # Input data
        input_token_ids = token_ids_input[len(list_input_data):len(list_input_data) + 256]
        list_input_data.append(input_token_ids)

        # Label data
        if idx == total_data - 1:
            label_token_ids = token_ids_input[len(list_input_data)+1:len(list_input_data) + 256]
            label_token_ids += [vocab_json["<EOS>"]]
        else:
            label_token_ids = token_ids_input[len(list_input_data)+1:len(list_input_data) + 256 + 1]

        list_label_data.append(label_token_ids)


    # Convert list into tensor
    input_data = torch.tensor(list_input_data)
    label_data = torch.tensor(list_label_data)
    

    return input_data, label_data

In [60]:
input_data, label_data = prepare_training_data(training_data)
input_data.size(), label_data.size()

(torch.Size([400705, 256]), torch.Size([400705, 256]))

In [61]:
torch.save(input_data, "input_data.pt")
torch.save(label_data, "label_data.pt")

In [69]:
a = torch.rand(10, 6, 256)
a.size()

torch.Size([10, 6, 256])

In [70]:
torch.argmax(a, dim=2)

tensor([[130,  18, 245, 213,  98, 246],
        [ 63,  71, 119, 171, 243, 242],
        [233, 183,  93, 248,  52, 139],
        [209, 134, 172, 109, 101, 142],
        [ 21,  14, 154,  59,   4,  27],
        [238, 117, 155, 231, 248, 213],
        [ 65,  89, 163,  48,  66, 137],
        [ 67, 124,  16, 240,  41,  84],
        [234, 115, 185, 101, 208, 149],
        [ 23,  23, 191, 170,  94, 190]])

In [78]:
a = torch.tensor([[1,2], [3,4]])
b = torch.tensor([[1,1], [3,4]])

(a==b).sum().item()

3

In [83]:
from pathlib import Path

current_path = Path(__file__).resolve()
current_path

NameError: name '__file__' is not defined

In [84]:
from transformers import AutoModelForCausalLM

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3b-instruct")
model

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (rotary_emb):

: 